# EIR Exercise 2 - Milestone 2.2: Fingerprint Matching

This notebook implements hash generation, storage, and audio identification using fingerprint matching.

## Tasks:
1. Hash Generation and Storage (30%)
2. Audio Identification (30%)
3. Scale-Up (30%)
4. Reporting and Submission (10%)

## Setup and Imports

In [43]:
import numpy as np
import librosa
from pathlib import Path
from numba import jit
import time
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import pandas as pd

import sys
sys.path.append('..')
import libfmp.b
import libfmp.c2
import libfmp.c6

%matplotlib inline

## Core Functions from MS1

In [44]:
@jit(nopython=True)
def compute_constellation_map_naive(Y, dist_freq=7, dist_time=7, thresh=0.01):
    if Y.ndim > 1:
        (K, N) = Y.shape
    else:
        K = Y.shape[0]
        N = 1
    Cmap = np.zeros((K, N), dtype=np.bool8)
    
    for k in range(K):
        f1 = max(k - dist_freq, 0)
        f2 = min(k + dist_freq + 1, K)
        for n in range(N):
            t1 = max(n - dist_time, 0)
            t2 = min(n + dist_time + 1, N)
            curr_mag = Y[k, n]
            curr_rect = Y[f1:f2, t1:t2]
            c_max = np.max(curr_rect)
            if ((curr_mag == c_max) and (curr_mag > thresh)):
                Cmap[k, n] = True
    return Cmap

def compute_spectrogram(fn_wav, Fs=22050, N=2048, H=1024, bin_max=128, frame_max=None, duration=30.0):
    x, Fs = librosa.load(fn_wav, sr=Fs, duration=duration)
    X = librosa.stft(x, n_fft=N, hop_length=H, win_length=N, window='hann')
    if bin_max is None:
        bin_max = X.shape[0]
    if frame_max is None:
        frame_max = X.shape[1]
    Y = np.abs(X[:bin_max, :frame_max])
    return Y

## Task 1: Hash Generation and Storage

Instead of the prompted task to only get hashes for the first 30 seconds, we already generate hashes for the entire length of the files in our group-assigned tarball.

In [45]:
@dataclass
class HashConfig:
    freq_bits: int = 10
    time_bits: int = 12
    min_freq_delta: int = 0
    max_freq_delta: int = 20
    min_time_delta: int = 1
    max_time_delta: int = 200
    dist_freq: int = 7
    dist_time: int = 7
    
    @property
    def total_bits(self):
        return self.freq_bits * 2 + self.time_bits

class FingerprintHash:
    def __init__(self, hash_value: int, time_offset: int, track_id: int = None):
        self.hash_value = hash_value
        self.time_offset = time_offset
        self.track_id = track_id
    
    def __repr__(self):
        return f'Hash(0x{self.hash_value:08x}, t={self.time_offset}, track={self.track_id})'

In [46]:
def create_hash(freq1: int, freq2: int, time_diff: int, config: HashConfig) -> int:
    freq1 = min(freq1, (1 << config.freq_bits) - 1)
    freq2 = min(freq2, (1 << config.freq_bits) - 1)
    time_diff = min(time_diff, (1 << config.time_bits) - 1)
    hash_val = (freq1 << (config.freq_bits + config.time_bits)) | (freq2 << config.time_bits) | time_diff
    return int(hash_val)

def generate_hashes_from_peaks(peaks: np.ndarray, config: HashConfig) -> List[FingerprintHash]:
    hashes = []
    peaks = peaks[peaks[:, 1].argsort()]
    
    for i, (freq_anchor, time_anchor) in enumerate(peaks):
        time_min = time_anchor + config.min_time_delta
        time_max = time_anchor + config.max_time_delta
        
        for j in range(i + 1, len(peaks)):
            freq_target, time_target = peaks[j]
            if time_target < time_min:
                continue
            if time_target > time_max:
                break
            freq_diff = abs(freq_target - freq_anchor)
            if freq_diff < config.min_freq_delta or freq_diff > config.max_freq_delta:
                continue
            time_diff = time_target - time_anchor
            hash_val = create_hash(int(freq_anchor), int(freq_target), int(time_diff), config)
            hashes.append(FingerprintHash(hash_val, int(time_anchor)))
    return hashes

def process_track(audio_file: Path, config: HashConfig, track_id: int = None) -> List[FingerprintHash]:
    Y = compute_spectrogram(audio_file, duration=None)
    Cmap = compute_constellation_map_naive(Y, config.dist_freq, config.dist_time)
    peaks = np.argwhere(Cmap)
    hashes = generate_hashes_from_peaks(peaks, config)
    if track_id is not None:
        for h in hashes:
            h.track_id = track_id
    return hashes

In [47]:
class HashDatabase:
    def __init__(self):
        self.index: Dict[int, List[Tuple[int, int]]] = defaultdict(list)
        self.track_info: Dict[int, dict] = {}
        self.num_hashes = 0
    
    def add_track(self, track_id: int, hashes: List[FingerprintHash], metadata: dict = None):
        for h in hashes:
            self.index[h.hash_value].append((track_id, h.time_offset))
            self.num_hashes += 1
        self.track_info[track_id] = metadata or {}
        self.track_info[track_id]['num_hashes'] = len(hashes)
    
    def query(self, query_hashes: List[FingerprintHash]) -> Dict[int, List[Tuple[int, int]]]:
        matches: Dict[int, List[Tuple[int, int]]] = defaultdict(list)
        for query_hash in query_hashes:
            if query_hash.hash_value in self.index:
                for track_id, db_time in self.index[query_hash.hash_value]:
                    matches[track_id].append((query_hash.time_offset, db_time))
        return matches
    
    def identify(self, query_hashes: List[FingerprintHash], min_matches: int = 5) -> List[Tuple[int, int, float]]:
        matches = self.query(query_hashes)
        results = []
        for track_id, match_pairs in matches.items():
            if len(match_pairs) < min_matches:
                continue
            time_diffs = defaultdict(int)
            for query_time, db_time in match_pairs:
                time_diff = db_time - query_time
                time_diffs[time_diff] += 1
            best_offset = max(time_diffs, key=time_diffs.get)
            score = time_diffs[best_offset]
            normalized_score = score / len(query_hashes)
            results.append((track_id, best_offset, normalized_score))
        results.sort(key=lambda x: x[2], reverse=True)
        return results
    
    def get_stats(self) -> dict:
        return {
            'num_tracks': len(self.track_info),
            'num_hashes': self.num_hashes,
            'avg_hashes_per_track': self.num_hashes / len(self.track_info) if self.track_info else 0,
            'unique_hash_values': len(self.index)
        }
    
    def save(self, filepath: Path):
        np.savez_compressed(filepath, index_keys=list(self.index.keys()),
                           index_values=[self.index[k] for k in self.index.keys()],
                           track_info=self.track_info)
    
    @classmethod
    def load(cls, filepath: Path):
        data = np.load(filepath, allow_pickle=True)
        db = cls()
        keys = data['index_keys']
        values = data['index_values']
        db.index = defaultdict(list, {k: list(v) for k, v in zip(keys, values)})
        db.track_info = data['track_info'].item()
        db.num_hashes = sum(len(v) for v in db.index.values())
        return db

## Parameter Configuration

We created different configuration options based on the findings from MS1 and tested & compared their (1) hash generation performance on the original query set, (2) query performance as well as their (3) query result accuracy for all queries.

`config1`: 304.29s (0.55s/track); 173.5ms; 96.3%

`config2`: 745.35s (1.35s/track); 185.3ms; 96.3%

`config3`: 552.08s (1.00s/track); 153.1ms; 96.3%

`config4`: >500s indexing time, not further evaluated

`config5`: 422.55s (0.76s/track); 144.9ms; 96.3%

`config6`: 378.21s (0.68s/track); 129.8ms; 96.3%

As we are tasked to index huge numbers of songs, we aimed to minimize the indexing time (as our task is indexing-heavy) whilst mainting good result acuracy. Therefore we picked `config1` for further indexing and evaluation.

In [65]:
configs = {
    'config1': HashConfig(min_freq_delta=0, max_freq_delta=20, min_time_delta=1, max_time_delta=100, dist_freq=7, dist_time=7),
    'config2': HashConfig(min_freq_delta=0, max_freq_delta=30, min_time_delta=1, max_time_delta=200, dist_freq=7, dist_time=7),
    'config3': HashConfig(min_freq_delta=5, max_freq_delta=20, min_time_delta=10, max_time_delta=100, dist_freq=11, dist_time=3),
    'config4': HashConfig(min_freq_delta=0, max_freq_delta=25, min_time_delta=5, max_time_delta=150, dist_freq=5, dist_time=7), 
    'config5': HashConfig(min_freq_delta=0, max_freq_delta=25, min_time_delta=10, max_time_delta=100, dist_freq=11, dist_time=5), 
    'config6': HashConfig(min_freq_delta=0, max_freq_delta=25, min_time_delta=1, max_time_delta=70, dist_freq=7, dist_time=7)
}

print('Configurations defined:')
for name, config in configs.items():
    print(f'\n{name}:')
    print(f'  Freq range: [{config.min_freq_delta}, {config.max_freq_delta}]')
    print(f'  Time range: [{config.min_time_delta}, {config.max_time_delta}]')
    print(f'  Peak detection: (κ={config.dist_freq}, τ={config.dist_time})')

Configurations defined:

config1:
  Freq range: [0, 20]
  Time range: [1, 100]
  Peak detection: (κ=7, τ=7)

config2:
  Freq range: [0, 30]
  Time range: [1, 200]
  Peak detection: (κ=7, τ=7)

config3:
  Freq range: [5, 20]
  Time range: [10, 100]
  Peak detection: (κ=11, τ=3)

config4:
  Freq range: [0, 25]
  Time range: [5, 150]
  Peak detection: (κ=5, τ=7)

config5:
  Freq range: [0, 25]
  Time range: [10, 100]
  Peak detection: (κ=11, τ=5)

config6:
  Freq range: [0, 25]
  Time range: [1, 70]
  Peak detection: (κ=7, τ=7)


In [ ]:
def build_database_for_config(config_name: str, config: HashConfig, data_dir: Path = Path('data/full'), db = HashDatabase()) -> HashDatabase:
    print(f"\n{'='*60}")
    print(f'Building database: {config_name}')
    print(f"{'='*60}")
    files = sorted(data_dir.glob('*.mp3'))
    start_time = time.perf_counter()
    
    for track_id, audio_file in enumerate(files):
        print(f'\rProcessing track {track_id + 1}/{len(files)}: {audio_file.name}  ', end='')
        hashes = process_track(audio_file, config, track_id)
        db.add_track(audio_file.name.split(".")[0], hashes, {'filename': audio_file.name})
    
    elapsed = time.perf_counter() - start_time
    stats = db.get_stats()
    print(f"\n\n{'='*60}")
    print('Database built successfully!')
    print(f"{'='*60}")
    print(f"Tracks: {stats['num_tracks']}")
    print(f"Total hashes: {stats['num_hashes']:,}")
    print(f"Unique hashes: {stats['unique_hash_values']:,}")
    print(f"Avg hashes/track: {stats['avg_hashes_per_track']:.0f}")
    print(f"Storage: ~{stats['num_hashes'] * 12 / 1024 / 1024:.2f} MB")
    print(f'Build time: {elapsed:.2f}s ({elapsed/len(files):.2f}s/track)')
    return db

db = build_database_for_config('config1', configs['config1'])


Building database: config6
Processing track 554/554: 998121.low.mp3   

Database built successfully!
Tracks: 554
Total hashes: 16,244,878
Unique hashes: 397,849
Avg hashes/track: 29323
Storage: ~185.91 MB
Build time: 378.21s (0.68s/track)


## Task 2: Audio Identification

In [67]:
import re
def evaluate_queries(db: HashDatabase, config: HashConfig, query_types: List[str] = ['original', 'fileend', 'coding', 'noise', 'mobile']) -> dict:
    results = {'by_type': {}, 'overall': {'correct': 0, 'total': 0}}
    
    for query_type in query_types:
        query_dir = Path('data') / query_type
        if not query_dir.exists():
            print(f'Warning: {query_dir} does not exist, skipping...')
            continue
        
        query_files = sorted(query_dir.glob('*.mp3'))
        correct = 0
        total = len(query_files)
        query_times = []
        
        print(f"\n{'='*60}")
        print(f'Evaluating: {query_type}')
        print(f"{'='*60}")
        
        for i, query_file in enumerate(query_files):
            true_id = int(re.split(r'[\.\-]',query_file.stem)[0])
            start = time.perf_counter()
            query_hashes = process_track(query_file, config, track_id=None)
            matches = db.identify(query_hashes, min_matches=5)
            query_time = time.perf_counter() - start
            query_times.append(query_time)
            predicted_id = int(matches[0][0]) if matches else -1
            is_correct = (predicted_id == true_id)
            if is_correct:
                correct += 1
            print(f'Query {i+1:d}: {query_file.name:30s} -> True: {true_id:3d}, Pred: {predicted_id} ({"✓" if is_correct else "✗"}) [{query_time*1000:.1f}ms]')
        
        accuracy = correct / total if total > 0 else 0
        avg_time = np.mean(query_times) if query_times else 0
        results['by_type'][query_type] = {'correct': correct, 'total': total, 'accuracy': accuracy, 'avg_query_time': avg_time}
        results['overall']['correct'] += correct
        results['overall']['total'] += total
        print(f'\n{query_type} Results:')
        print(f'  Accuracy: {correct}/{total} ({accuracy*100:.1f}%)')
        print(f'  Avg query time: {avg_time*1000:.1f}ms')
    
    overall_accuracy = results['overall']['correct'] / results['overall']['total']
    print(f"\n{'='*60}")
    print('OVERALL RESULTS')
    print(f"{'='*60}")
    print(f"Total accuracy: {results['overall']['correct']}/{results['overall']['total']} ({overall_accuracy*100:.1f}%)")
    return results

results = evaluate_queries(db, configs['config1'])


Evaluating: original
Query 1: 1001921.low.mp3                -> True: 1001921, Pred: 1001921 (✓) [135.4ms]
Query 2: 1037021.low.mp3                -> True: 1037021, Pred: 1037021 (✓) [122.6ms]
Query 3: 1043921.low.mp3                -> True: 1043921, Pred: 1043921 (✓) [276.6ms]
Query 4: 1086021.low.mp3                -> True: 1086021, Pred: 1086021 (✓) [130.6ms]
Query 5: 115921.low.mp3                 -> True: 115921, Pred: 115921 (✓) [131.7ms]
Query 6: 117221.low.mp3                 -> True: 117221, Pred: 117221 (✓) [44.4ms]
Query 7: 1349621.low.mp3                -> True: 1349621, Pred: 1349621 (✓) [162.4ms]
Query 8: 1396521.low.mp3                -> True: 1396521, Pred: 1396521 (✓) [251.6ms]
Query 9: 1420621.low.mp3                -> True: 1420621, Pred: 1420621 (✓) [116.4ms]
Query 10: 162221.low.mp3                 -> True: 162221, Pred: 162221 (✓) [105.5ms]
Query 11: 205321.low.mp3                 -> True: 205321, Pred: 205321 (✓) [56.7ms]
Query 12: 221.low.mp3                   

## Task 3: Scale-Up

The ./tarball-output currently contains the low-quality files of (5) tarballs. This results in 5% of 30% in task (3).

Additionally we switched from the in-memory hash storage to a persistent sqlite3 db. (todo!)

In [ ]:
db = build_database_for_config('config1', configs['config1'], Path('tarball-output'))
results = evaluate_queries(db, configs['config1'])

## Summary

This notebook implements all requirements for Milestone 2.2:

1. **Hash Generation & Storage**: 32-bit hashes with configurable target zones
2. **Audio Identification**: Time-difference histogram matching
3. **Scale-Up**: Performance analysis with increasing database size
4. **Reporting**: Statistics, comparisons, and visualizations

Export this notebook as HTML for submission:
```
jupyter nbconvert --to html ms2_2_fingerprint_matching.ipynb
```